## Assignment 2 - Mortality in the ICU

In this assignment we are going to build a predictive model for mortality in ICU patients, using a database of 1010 patients and various measurements taken at admission and throughout their first 7 days of their stay at the UMCG ICU (courtesy of Dr Eric Keus).  Mortality outcomes at ICU, and at several time points afterwards, are included.

This problem is different from the one we tackled in the first part of this week, in various ways.  First, the target variable is not a continuous variable (heart valve size), but a binary outcome (mortality).  This is therefore a *classification* problem.  In keeping with the approach taken earlier, we will use a classical model to tackle this problem, namely, logistic regression.

There are other differences as well.  There are now very many more predictive variables available, increasing the risk of over-fitting.  Therefore, regularization is important (and cross validation remains important to properly assess the quality of our models).

The dataset is also somewhat messy - again, this is actual data, and this messiness is to be expected.  Some variables have very few entries, and should be skipped.  Some variables pertain to days 2 and onwards; we will focus on the prediction problem at admission, and we will not use those.  Many variables have at least some missing data, and we will need to deal with that.

<div class="alert alert-success">

## Assignment

* Your overall task is to build the **best predictive model** you can find for 7-day ICU mortality
* As figure of merit, use the area under the ROC curve ("roc_auc")
* To make things comparable, use simple 5-fold cross validation
* Deadline for submission of (runnable, clearly annotated with running English text) .ipynb: **Thursday 17:00**
* You are asked to work in pairs.  Please include **both your names** in the filename of the notebook you submit.

While you're entirely free to come up with your own strategy, here are some ideas:

* Start with logistic regression, and use a validation curve to find the optimal regularization strength
* Consider adding quadratic and mixed terms for the important parameters
* Try some tree-based and other classifiers implemented in Scikit-learn
* Try XGBoost - you will (probably) need to install it first.  Try modifying some hyperparameters.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()

Only run the code below when you use `Google Colab`.  

In [ ]:
## Only run this when you use Google Colab.
## Otherwise, make sure 'icu_data.csv' is located in your Jupyter directory; or edit the path below

from google.colab import files
## Upload 'icu_data.csv'
files.upload();

Run the code below to load the data.  We remove all columns pertaining to measurements after the first day of admission, and we also remove the meaningless index column

If you use Jupyter, make sure 'icu_data.csv' is located in your Jupyter directory;
or else edit the path below to point it to where the file is located

In [ ]:
df = pd.read_csv("icu_data.csv")

to_remove = ["Unnamed: 0"]
for var in df.columns:
    if var.find('_d') > -1 and var.find('_d1') == -1:
        to_remove.append(var)
df = df.drop( columns = to_remove )

print(df.info())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   age                     1010 non-null   int64  
 1   gender                  1010 non-null   int64  
 2   height                  973 non-null    float64
 3   weight                  1010 non-null   float64
 4   bsa                     973 non-null    float64
 5   diabetes_sics           1010 non-null   int64  
 6   hypertension            1010 non-null   int64  
 7   previousmi              1010 non-null   int64  
 8   previousrenalfailure    1010 non-null   int64  
 9   confirmedinfection      1010 non-null   int64  
 10  gcs_d1                  958 non-null    float64
 11  pao2_d1                 937 non-null    float64
 12  fio2_d1                 958 non-null    float64
 13  mech_vent_d1            958 non-null    object 
 14  special_vent_method_d1  957 non-null    

,age,gender,height,weight,bsa,diabetes_sics,hypertension,previousmi,previousrenalfailure,confirmedinfection,...,cv_d1_SOFA,gcs_d1_sofa,creat_d1_SOFA,tot_d1_SOFA,max_SOFA,mort_icu,mort7,mort30,mort90,mort180
count,1010.000000,1010.000000,973.000000,1010.000000,973.000000,1010.000000,1010.000000,1010.000000,1010.000000,1010.000000,...,939.000000,994.000000,964.000000,810.000000,916.000000,1010.000000,1009.000000,1000.000000,999.000000,999.000000
mean,60.628713,1.623762,175.591675,81.247921,1.963015,1.188119,1.305941,1.129703,1.091089,1.192079,...,1.826411,0.382294,0.475104,4.895062,6.271834,1.163366,1.133796,1.220000,0.263263,0.286286
std,15.443422,0.484681,9.974955,17.942156,0.229557,0.391001,0.461033,0.336143,0.287878,0.394130,...,1.353628,1.022766,0.805229,2.711072,2.999328,0.369883,0.340602,0.414454,0.440625,0.452251
min,18.000000,1.000000,125.000000,41.000000,1.317000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,52.000000,1.000000,169.000000,70.000000,1.810000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,0.000000,0.000000,3.000000,4.000000,1.000000,1.000000,1.000000,0.000000,0.000000
50%,64.000000,2.000000,175.000000,80.000000,1.966000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,0.000000,0.000000,5.000000,6.000000,1.000000,1.000000,1.000000,0.000000,0.000000
75%,72.000000,2.000000,183.000000,90.250000,2.106000,1.000000,2.000000,1.000000,1.000000,1.000000,...,3.000000,0.000000,1.000000,6.000000,8.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,90.000000,2.000000,207.000000,168.000000,2.677000,2.000000,2.000000,2.000000,2.000000,2.000000,...,4.000000,4.000000,3.000000,16.000000,16.000000,2.000000,2.000000,2.000000,1.000000,1.000000


First identify the target variable, 7-day mortality, which is coded in the column `mort7`.  There are other outcome variables as well - this includes all other variables starting with `mort`, and `max_SOFA`.  Remove these.

The data has several small issues.  There is one variable with huge missingness - remove this.  Records where the outcome is missing should also be removed.  

Several other variables show a degree of missingness; use imputation to deal with this.